In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

FILE = "Water Quality.csv"  # Set to full path if needed
OUT = "task3_timeseries_outputs"
TOP_PARAMETERS = 8
ROLLING_MONTHS = 3

def main():
    if not Path(FILE).exists():
        raise FileNotFoundError(f"Cannot find {FILE!r}. Edit FILE to the full path of Water Quality.csv.")
    Path(OUT).mkdir(exist_ok=True)
    df = pd.read_csv(FILE, low_memory=False)
    req = {"ActivityStartDate","CharacteristicName","ResultValue"}
    if req-set(df.columns): raise ValueError(f"Missing columns: {req-set(df.columns)}")
    df["ActivityStartDate"] = pd.to_datetime(df["ActivityStartDate"], errors="coerce")
    df["ResultValue"] = pd.to_numeric(df["ResultValue"], errors="coerce")
    df = df.dropna(subset=["ActivityStartDate","CharacteristicName","ResultValue"])
    if df.empty: raise ValueError("No valid date/numeric measurements found.")
    top = df.CharacteristicName.value_counts().head(TOP_PARAMETERS).index
    fig, summary = go.Figure(), []
    for param in top:
        monthly = df[df.CharacteristicName==param].set_index("ActivityStartDate").ResultValue.resample("MS").median().dropna()
        if monthly.empty: continue
        fig.add_trace(go.Scatter(x=monthly.index,y=monthly.values,mode="lines+markers",name=f"{param} — monthly median"))
        roll = monthly.rolling(ROLLING_MONTHS,min_periods=1).mean()
        fig.add_trace(go.Scatter(x=roll.index,y=roll.values,mode="lines",name=f"{param} — {ROLLING_MONTHS}-month rolling mean",line=dict(dash="dash")))
        if len(monthly)>=2:
            x=np.arange(len(monthly)); slope, intercept=np.polyfit(x,monthly.values,1)
            fig.add_trace(go.Scatter(x=monthly.index,y=intercept+slope*x,mode="lines",name=f"{param} — linear trend",line=dict(dash="dot")))
            summary.append({"CharacteristicName":param,"monthly_points":len(monthly),
                            "start_month":str(monthly.index.min().date()),"end_month":str(monthly.index.max().date()),
                            "linear_slope_per_month_step":slope,"direction":"upward" if slope>0 else "downward" if slope<0 else "flat"})
    fig.update_layout(title="AquaInsight — Interactive Water Quality Time Series",xaxis_title="Date",
                      yaxis_title="ResultValue (original units)",hovermode="x unified",template="plotly_white",height=750)
    fig.write_html(Path(OUT)/"interactive_water_quality_timeseries.html",include_plotlyjs=True)
    pd.DataFrame(summary).to_csv(Path(OUT)/"linear_trend_summary.csv",index=False)
    print(f"Saved interactive chart and trend summary in {OUT}; open the HTML file in a browser.")
    print("Different parameters may have different units/scales; compare each parameter's trend separately.")
if __name__=="__main__": main()


Saved interactive chart and trend summary in task3_timeseries_outputs; open the HTML file in a browser.
Different parameters may have different units/scales; compare each parameter's trend separately.
